# Qwen Image 2.1 — Colabで透過画像を生成
上から順に実行します。「ランタイム → ランタイムのタイプを変更 → GPU」を選択してください。

公式デモの待ち行列は使用しません。モデルをダウンロードし、このColabのGPUで実行します。必要なVRAM/RAMは実機未検証です。無料GPUでの動作は保証できません。初回ダウンロードには時間・ディスク容量が必要です。

このノートブックは対話的な画像生成用です。公開サイト向けAPIの常設運用には専用GPUを使用してください。有料GPUの利用はご自身のアカウントで確認してください。

[公式README](https://github.com/QwenLM/Qwen-Image-2.1) · [ライセンス](https://github.com/QwenLM/Qwen-Image-2.1/blob/main/LICENSE)


In [ ]:
%pip install "torch>=2.4.0" "transformers>=5.17" accelerate pillow psutil
%pip install git+https://github.com/huggingface/diffusers


インストール後に再起動を求められた場合は、ランタイムを再起動して次のセルから進めてください。

In [ ]:
import torch, psutil, shutil
assert torch.cuda.is_available(), 'GPUランタイムを選択してください。'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print('RAM (GB):', round(psutil.virtual_memory().total / 2**30, 1))
print('Disk free (GB):', round(shutil.disk_usage('/content').free / 2**30, 1))


## モデルを読み込む
CPUオフロードを初期設定にしています。GPUメモリを節約しますが、通常のRAMと転送時間が必要です。T4などBF16非対応GPUではFP16を使用しますが、動作・品質は未検証です。

In [ ]:
import os
import torch
from diffusers import QwenImage21Pipeline


def load_pipeline():
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA GPUが必要です。GPUランタイムを選択してください。')
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    pipe = QwenImage21Pipeline.from_pretrained('Qwen/Qwen-Image-2.1', torch_dtype=dtype)
    if os.environ.get('QWEN_CPU_OFFLOAD', '1') == '1':
        pipe.enable_model_cpu_offload()
    else:
        pipe.to('cuda')
    return pipe


def generate(pipe, prompt, size=1024, seed=42, steps=40, images=None):
    kwargs = dict(prompt=prompt, width=size, height=size, num_inference_steps=steps,
                  generator=torch.Generator('cuda').manual_seed(seed))
    if images:
        kwargs['image'] = images
    with torch.inference_mode():
        return pipe(**kwargs).images[0]

pipe = load_pipeline()


## 画像生成
まず1024pxで試し、余裕があれば2048pxに変更してください。透過は結果のアルファ値を確認します。

In [ ]:
prompt = "This is an RGBA image with transparency. A cute cartoon dragon sticker. The image has alpha channel and the background is transparent."
size = 1024
seed = 42
steps = 40
image = generate(pipe, prompt, size, seed, steps)
image.save('/content/qwen-transparent.png')
display(image)
print('Mode:', image.mode, 'Size:', image.size)
print('Alpha range:', image.getchannel('A').getextrema() if 'A' in image.getbands() else 'No alpha channel')


In [ ]:
from google.colab import files
files.download('/content/qwen-transparent.png')


## 任意：参照画像で編集
ここからは編集を試したい場合だけ実行してください。アップロードした画像はこのColab環境で処理します。

In [ ]:
from PIL import Image
uploaded = files.upload()
assert 1 <= len(uploaded) <= 10, '参照画像は1〜10枚です。'
references = [Image.open(name) for name in uploaded]
edit_prompt = "This is an RGBA image with transparency. Extract the main subject and preserve its details. The image has alpha channel and the background is transparent."
edited = generate(pipe, edit_prompt, size, seed, steps, references)
edited.save('/content/qwen-edited.png')
display(edited)
files.download('/content/qwen-edited.png')


## メモリ不足の場合
1024px・参照画像1枚で試してください。CPUオフロードでも足りない場合は、GPUメモリと通常RAMに余裕がある環境へ変更する必要があります。終了後は「ランタイム → 接続を解除してランタイムを削除」でGPUを解放してください。